In [256]:
import requests
import polars as pl


In [257]:
data = requests.get("https://ogd-static.voteinfo-app.ch/v4/ogd/kommunale_resultate_2026_03_08.json").json()

In [258]:
results = list(filter(lambda row: row['vorlagenId']==369492, data['kantone'][1]['vorlagen']))[0]

In [259]:
mapping_str = """7fff0265-2bda-4e41-a45d-996b4a9a41c3/Serap Kahriman
927ddfc7-0bbd-4b51-aa0e-ce4e8e064ac4/Raphael Golta
994f4a04-97b4-4433-959e-deecb5f12c2f/Përparim Avdili
9a6b22bd-a93d-4d29-aa3d-d528bcb1a9d0/Ueli Bamert
9ef04264-9fbf-425c-9c50-3e99a7939ffe/Andere Kandidierende
d5b5d297-f6af-4f99-b170-7d73c9ebea04/Andere Kandidierende
999/Andere Kandidierende"""

mapping = dict(map(lambda line: line.split("/"), mapping_str.split("\n")))


In [260]:
mapping

{'7fff0265-2bda-4e41-a45d-996b4a9a41c3': 'Serap Kahriman',
 '927ddfc7-0bbd-4b51-aa0e-ce4e8e064ac4': 'Raphael Golta',
 '994f4a04-97b4-4433-959e-deecb5f12c2f': 'Përparim Avdili',
 '9a6b22bd-a93d-4d29-aa3d-d528bcb1a9d0': 'Ueli Bamert',
 '9ef04264-9fbf-425c-9c50-3e99a7939ffe': 'Andere Kandidierende',
 'd5b5d297-f6af-4f99-b170-7d73c9ebea04': 'Andere Kandidierende',
 '999': 'Andere Kandidierende'}

In [261]:

rows = []
for zaehlkreis in results['zaehlkreise']:
    for kandidat in zaehlkreis['resultat']['kandidaten']:
        rows.append({
            "zaehlkreis": zaehlkreis['geoLevelname'],
            "kandidat": mapping[kandidat['kandidatNummer']],
            "stimmen": kandidat['stimmen']
        })

df = pl.DataFrame(rows)

In [262]:
order = """Raphael Golta
Përparim Avdili
Ueli Bamert
Serap Kahriman
Andere Kandidierende
""".split('\n')

df_order = pl.DataFrame({
    "kandidat": order,
    "order": range(len(order))
})

In [263]:
df_wide = df.pivot(index="kandidat", on="zaehlkreis", values="stimmen", aggregate_function="sum")

In [264]:
df_wide

kandidat,Zürich Kreise 1 und 2,Zürich Kreis 3,Zürich Kreis 4 und 5,Zürich Kreis 6,Zürich Kreise 7 und 8,Zürich Kreis 9,Zürich Kreis 10,Zürich Kreis 11,Zürich Kreis 12
str,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""Serap Kahriman""",0,1282,0,0,0,0,1308,1451,400
"""Raphael Golta""",0,7643,0,0,0,0,6840,6442,2151
"""Përparim Avdili""",0,2982,0,0,0,0,2924,3401,1131
"""Andere Kandidierende""",0,814,0,0,0,0,733,1003,328
"""Ueli Bamert""",0,1359,0,0,0,0,1657,2891,1154


In [265]:
df_final = df_order.join(df_wide, on='kandidat', how='left').sort('order').drop('order')
df_final

kandidat,Zürich Kreise 1 und 2,Zürich Kreis 3,Zürich Kreis 4 und 5,Zürich Kreis 6,Zürich Kreise 7 und 8,Zürich Kreis 9,Zürich Kreis 10,Zürich Kreis 11,Zürich Kreis 12
str,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""Raphael Golta""",0,7643,0,0,0,0,6840,6442,2151
"""Përparim Avdili""",0,2982,0,0,0,0,2924,3401,1131
"""Ueli Bamert""",0,1359,0,0,0,0,1657,2891,1154
"""Serap Kahriman""",0,1282,0,0,0,0,1308,1451,400
"""Andere Kandidierende""",0,814,0,0,0,0,733,1003,328
"""""",null,null,null,null,null,null,null,null,null


In [266]:
df_final.write_excel("results.xlsx")